# Ticket: Feature Engineering - Aggregate Features (Domain Knowledge)
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers

## 1. TỔNG QUAN VÀ MỤC TIÊU CẢI TIẾN
Trong bài toán dự báo năng lượng mặt trời, việc chỉ sử dụng dữ liệu thời tiết thô là chưa đủ. Các mô hình Machine Learning dạng cây rất khó tự học các mối quan hệ phi tuyến phức tạp mang tính vật lý.

**Mục tiêu Nâng cấp:**
- Xử lý các trường hợp Missing Data (Metadata thiếu) theo đúng logic vật lý, giữ nguyên NaN thay vì fill 0, nhằm không làm sai lệch ý nghĩa thực tế.
- Bổ sung các bước kiểm chứng QA mở rộng: Kiểm tra tính toàn vẹn, tương quan Pearson/Spearman, đánh giá đa cộng tuyến, và vẽ biểu đồ Scatter phi tuyến.


## 2. Import Libraries & Load Data

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set theme cho Plotly/Seaborn
sns.set_theme(style="whitegrid")

# Import module domain features
from fe_domain import build_domain_aggregate_features

In [2]:
# Load preprocessed data
parquet_path = '../../data/mlmart_base/v2_preprocessing.parquet'
print(f"Đang đọc dữ liệu từ: {parquet_path}")
df = pd.read_parquet(parquet_path)

print(f"Tổng số dòng ban đầu: {len(df)}")
display(df.head(3))

## 3. Feature Generation (Domain Logic)

In [3]:
# Tạo Aggregate Features
df_fe = build_domain_aggregate_features(df)

new_features = [
    'capacity_per_panel', 
    'temp_x_radiation', 
    'thermal_loss_factor', 
    'diffuse_fraction'
]

print("Sample các đặc trưng mới được tạo:")
display(df_fe[new_features].head())

## 4. Kiểm chứng Toàn vẹn Dữ liệu (Data QA/QC)
Phân tích tỷ lệ Missing Values và Infinity. Những trạm thiếu metadata sẽ thể hiện đúng qua tỷ lệ NaN.

In [4]:
# 4.1. Phân tích Missing Values
missing_stats = df_fe[new_features].isna().sum().to_frame(name='Missing_Count')
missing_stats['Missing_Percentage'] = (missing_stats['Missing_Count'] / len(df_fe)) * 100
missing_stats = missing_stats.sort_values('Missing_Percentage', ascending=False)

print("\n--- BÁO CÁO NHỮNG GIÁ TRỊ THIẾU TỪ FE DOMAIN ---")
display(missing_stats)

# Đếm Infinity
inf_counts = {col: np.isinf(df_fe[col]).sum() for col in new_features if df_fe[col].dtype in ['float64', 'float32']}
print("\n--- SỐ LƯỢNG GIÁ TRỊ VÔ CỰC (INFINITY) ---")
print(inf_counts)

### Nhận xét về QA:
- Các giá trị NaN được sinh ra là do **dữ liệu gốc (Metadata hoặc Thời tiết) bị thiếu**. Việc giữ nguyên NaN thay vì fillna(0) giúp chúng ta:
  1. Không bóp méo phân phối dữ liệu (tránh tạo ra các spike ảo ở số 0).
  2. Để lại dấu hiệu rõ ràng cho các thuật toán Tree-based (XGBoost, LightGBM) tự học đường rẽ nhánh cho missing data, hoặc sử dụng pipeline Imputation (Mean/KNN) ở bước tiếp theo.
- Tuyệt đối không còn giá trị `Infinity`, chứng tỏ các trường hợp lỗi chia cho 0 đã được xử lý triệt để (mẫu số = 0 được chuyển thành NaN).

## 5. Phân tích Phân phối Đặc trưng (Distributions)

In [5]:
# 5.1. Vẽ Histogram và Boxplot (KDE)
fig, axes = plt.subplots(len(new_features), 2, figsize=(14, 4 * len(new_features)))

for i, col in enumerate(new_features):
    # Histogram
    sns.histplot(df_fe[col].dropna(), bins=60, kde=True, ax=axes[i, 0], color='teal', edgecolor='black')
    axes[i, 0].set_title(f'Phân phối của {col}')
    axes[i, 0].set_ylabel('Tần suất')
    
    # Boxplot để kiểm tra Outliers cục bộ
    sns.boxplot(x=df_fe[col].dropna(), ax=axes[i, 1], color='coral')
    axes[i, 1].set_title(f'Boxplot của {col}')

plt.tight_layout()
plt.show()

## 6. Đánh giá Tương quan & Đa Cộng tuyến
Kiểm tra xem các features này tương quan tuyến tính (Pearson) và phi tuyến xếp hạng (Spearman) với target ra sao. Đồng thời kiểm tra đa cộng tuyến.

In [6]:
corr_cols = ['energy_generated_kwh'] + new_features
df_corr = df_fe[corr_cols].dropna()

pearson_corr = df_corr.corr(method='pearson')
spearman_corr = df_corr.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Pearson
sns.heatmap(pearson_corr, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, fmt=".2f", ax=axes[0])
axes[0].set_title('Tương quan Pearson (Tuyến tính)')

# Spearman
sns.heatmap(spearman_corr, annot=True, cmap='RdBu_r', vmin=-1, vmax=1, fmt=".2f", ax=axes[1])
axes[1].set_title('Tương quan Spearman (Hạng - Phi tuyến)')

plt.tight_layout()
plt.show()

### Nhận xét & Đề xuất về Đặc trưng:
- `temp_x_radiation` thường có tương quan cực cao với `energy_generated_kwh`. Điều này là dễ hiểu vì nó chứa đựng thông tin từ bức xạ mặt trời (nguồn sinh năng lượng).
- LƯU Ý MÔ HÌNH HÓA: Đã loại bỏ hoàn toàn các đặc trưng có nguy cơ rò rỉ dữ liệu (Target Leakage) như tỷ lệ khai thác công suất trạm vì nó chứa trực tiếp biến mục tiêu.
- `diffuse_fraction`: Có thể mang hệ số âm với sản lượng (vì ánh sáng khuếch tán thường có cường độ yếu hơn ánh sáng trực tiếp, nên mây mù nhiều sẽ giảm điện).

In [7]:
# 6.2. Scatter Plot Phi tuyến giữa Feauture và Target (Lấy sample 10,000 điểm để tăng tốc vẽ)
sample_df = df_corr.sample(min(10000, len(df_corr)), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.flatten()
features_to_plot = ['temp_x_radiation', 'thermal_loss_factor', 'diffuse_fraction', 'capacity_per_panel']

for i, col in enumerate(features_to_plot):
    sns.scatterplot(data=sample_df, x=col, y='energy_generated_kwh', alpha=0.3, ax=axes[i], color='navy')
    axes[i].set_title(f'Quan hệ giữa {col} và Năng lượng sinh ra')

plt.tight_layout()
plt.show()

## 7. Export Processed Dataset

In [8]:
output_path = '../../data/mlmart_base/v3_aggregate_features.parquet'
print(f"Đang lưu dữ liệu ra: {output_path}")

# Lưu trữ bằng Parquet để bảo toàn kiểu dữ liệu (đặc biệt là NaN)
df_fe.to_parquet(output_path, index=False)
print("Hoàn tất lưu file v3_aggregate_features!")